In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
import cv2
import numpy as np  
import open3d as o3d
import time
import utils.visuals

from utils.constants import *
from utils.paths import *
from utils.pcd_proccessing import *
from utils.visuals import *


In [17]:
# функция для отсчета времени
def timer(info):
    global cur_time
    print(f"{info}: {-(cur_time - (cur_time := time.time())):.4f} секунд")

In [18]:
# pcd = o3d.io.read_point_cloud("test_pcd.ply")
pcd_obj = o3d.data.PLYPointCloud()
pcd = o3d.io.read_point_cloud(pcd_obj.path)

In [19]:
print(f"Исходное количество точек: {len(pcd.points)}")

start = time.time()
cur_time = start

downsampled_pcd = downsample(pcd)

timer("Даунсемплинг(случайный)")

#Ищем плоскость пола и точки в ней и вне неё
plane_model, _ = downsampled_pcd.segment_plane(distance_threshold = distance_threshold,
                                               num_iterations = ransac_iterations,
                                               ransac_n = 3)
[a, b, c, d] = plane_model
#строим нормаль к поверхности   
normal = np.array([a, b, c])

timer("Поиск плоскости пола")

# Убираем слишком далекие от пола точки
pcd_points = np.asarray(pcd.points)
close_mask = np.abs(a * pcd_points[:, 0] 
                  + b * pcd_points[:, 1] 
                  + c * pcd_points[:, 2] 
                  + d) <= max_dist
close_points = pcd_points[close_mask]

timer("Выбор близких к полу точек")

close_points_ds = voxel_downsample_fast(close_points, grid_step)

timer("Voxel-Даунсемплинг близких точек")

# Разделяем на inliers и outliers
close_distances_ds = np.abs(np.dot(close_points_ds, normal) + d)
inlier_mask = close_distances_ds <= distance_threshold
inliers = close_points_ds[inlier_mask]
outliers = close_points_ds[~inlier_mask]

timer("Выделение точек в и вне плоскости")

#меняем направление нормали в сторону большего количества точек
condition = np.dot(outliers, normal) + d < 0
if len(outliers[condition]) > len(outliers) // 2:
    normal = -normal

# берем 2 перпендикулярных вектора в плоскости
v1_norm = np.linalg.norm([b, -a, 0])
plane_v1 = np.asarray([b, -a, 0]) / v1_norm
plane_v2 = np.linalg.cross(normal, plane_v1)

#проецируем точки пола на плоскость пола
projected = project_to_plane(inliers, plane_model)

timer("Проекция точек пола на плоскость")

#переходим от координат в пространстве в координаты 2-х векторов в плоскости
plane_coords, x_max, x_min, y_max, y_min = switch_to_plane_coords(projected, plane_v1, plane_v2)

#строим двумерный массив, соответствующий плоскости
grid = np.zeros((
    int((y_max - y_min) // grid_step + 1), 
    int((x_max - x_min) // grid_step + 1)
))
# Инвертируем Y т.к. в opencv 0 по оси Y это самый верх
x_indices = ((plane_coords[:, 0] - x_min) // grid_step).astype(int)
y_indices = ((y_max - plane_coords[:, 1]) // grid_step).astype(int) 
valid_mask = (x_indices >= 0) & (x_indices < grid.shape[1]) & \
             (y_indices >= 0) & (y_indices < grid.shape[0])

grid[y_indices[valid_mask], x_indices[valid_mask]] = 1

# наращиваем точки пола
kernel = np.ones((morph_fill_size,morph_fill_size))
grid = cv2.morphologyEx(grid, cv2.MORPH_CLOSE, kernel)
floor_grid = np.copy(grid)

timer("Составление сетки пола")

# проецируем точки вне пола на плоскость
projected_outliers = project_to_plane(outliers, plane_model)

timer("Проекция точек препятствия на пол")

# убираем из плоскости точки, где есть препятствия
outliers_plane_coords = switch_to_plane_coords(projected_outliers, plane_v1, plane_v2)[0]
x_indices = ((outliers_plane_coords[:, 0] - x_min) // grid_step).astype(int)
y_indices = ((y_max - outliers_plane_coords[:, 1]) // grid_step).astype(int)
valid_mask = (x_indices >= 0) & (x_indices < grid.shape[1]) & \
             (y_indices >= 0) & (y_indices < grid.shape[0])

floor_obstacles = grid
floor_obstacles[y_indices[valid_mask], x_indices[valid_mask]] = 0

# наращиваем точки пола
kernel = np.ones((morph_size,morph_size))
floor_obstacles = cv2.morphologyEx(floor_obstacles, cv2.MORPH_OPEN, kernel)

timer("Составление сетки пола с препятствиями")
print(f"Общее время: {time.time() - start:.4f} секунд")

Исходное количество точек: 196133
Даунсемплинг(случайный): -0.0000 секунд
Поиск плоскости пола: 0.1796 секунд
Выбор близких к полу точек: 0.0149 секунд
Voxel-Даунсемплинг близких точек: 0.0290 секунд
Выделение точек в и вне плоскости: 0.0054 секунд
Проекция точек пола на плоскость: 0.0046 секунд
Составление сетки пола: 0.0079 секунд
Проекция точек препятствия на пол: 0.0046 секунд
Составление сетки пола с препятствиями: 0.0077 секунд
Общее время: 0.2538 секунд


In [20]:
floor_accessible = np.copy(floor_obstacles)

#убираем маленькие компоненты 
components_info = cv2.connectedComponentsWithStats(
    floor_accessible.astype(np.uint8), 
    connectivity=4, 
    ltype=cv2.CV_32S
)
[num_labels, labels, stats, centroids] = components_info
for i in range(1, num_labels):
    area = stats[i, cv2.CC_STAT_AREA]*grid_step**2
    if area < min_area:
        floor_accessible[labels == i] = 0

# сначала жестко так прям убираем шум
kernel = np.ones((morph_fill_size, morph_fill_size))
floor_polished = cv2.morphologyEx(floor_accessible, cv2.MORPH_CLOSE, kernel)

floor_expanded = np.copy(floor_polished)

# теперь возвращаем препятствия, которые стерлись как шум
floor_accessible_reversed = 1 - floor_accessible
floor_polished_reversed = 1 - floor_expanded
components_info = cv2.connectedComponentsWithStats(
    floor_accessible_reversed.astype(np.uint8), 
    connectivity=4, 
    ltype=cv2.CV_32S
)
[num_labels, labels, stats, centroids] = components_info
for i in range(1, num_labels):
    obstacle = floor_accessible_reversed[labels == i]
    if np.any(cv2.bitwise_and(obstacle, floor_polished_reversed[labels == i])):
        floor_expanded[labels == i] = 0

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(diameter/grid_step), int(diameter/grid_step)))
floor_walkable = cv2.erode(floor_expanded, kernel)

# pcd_thread = show_pcd_concurrent(pcd)
# im_thread = show_images_parallel([floor_obstacles, floor_polished, floor_expanded])
# im_thread.join()
# pcd_thread.join()

pathvis = PathVisualizer(floor_walkable, floor_expanded)

In [23]:
pathvis.find_path()

huy


In [22]:
# unused visualization

# inliers_pcd = o3d.geometry.PointCloud()
# inliers_pcd.points = o3d.utility.Vector3dVector(inliers)
# inliers_pcd.paint_uniform_color([0, 1, 0])

# outliers_pcd = o3d.geometry.PointCloud()
# outliers_pcd.points = o3d.utility.Vector3dVector(outliers)
# outliers_pcd.paint_uniform_color([1, 0, 0])

# visualise_concurrent(inliers_pcd)
# visualise_concurrent(outliers_pcd)
# visualise(projected)
# visualise(projected_outliers)

# # Точки близкие к полу
# visualise(close_points)

# #выделенная плоскость
# inliers_pcd = o3d.geometry.PointCloud()
# inliers_pcd.points = o3d.utility.Vector3dVector(inliers)

# outliers_pcd = o3d.geometry.PointCloud()
# outliers_pcd.points = o3d.utility.Vector3dVector(outliers)

# inliers_pcd.paint_uniform_color([0, 1, 0])
# outliers_pcd.paint_uniform_color([1, 0, 0])
# vis.draw_geometries([inliers_pcd, outliers_pcd])

# #проекция точек пола на плоскость пола
# visualise(projected)

# # Проекция точек вне пола на плоскость пола
# visualise(projected_outliers)
